### Tampering Taxonomy

| Fixture Type | What is Modified | Expected Verdict | Expected Secret (DJ Oracle) |
| :--- | :--- | :--- | :--- |
| **Authentic** | Nothing (Baseline) | Authentic | All zeros (Constant) |
| **Data Tampered** | Data changed, old tag kept | Tampered | Non-zero (Balanced) |
| **Nonce Tampered** | Nonce changed, old tag kept | Tampered | Non-zero (Balanced) |
| **Tag Tampered** | Tag changed directly | Tampered | Non-zero (Balanced) |
| **Forged (Wrong Key)** | Attacker signs with unshared key | Tampered | Non-zero (Balanced) |
| **Corrupted** | Random bit flips in payload | Tampered or Parse-error | Non-zero / N/A |

In [1]:
import sys
import os

# Add the parent directory to Python's search path
sys.path.append(os.path.abspath('..'))

In [3]:

import json
from quantum_qr.config import get_key
from quantum_qr.qr_io import read_qr_code
from quantum_qr.payload import (
    decode_payload, 
    compute_tag, 
    tags_to_secret
)

# Adjust this if you saved your fixtures to a different path
out_dir = "../data/fixtures"
manifest_path = os.path.join(out_dir, "manifest.json")

# Load the manifest
with open(manifest_path, "r") as f:
    manifest = json.load(f)

# Use the exact same key and parameters used during generation
key = get_key()
n_bits = 8

print(f"🔍 Sanity checking {len(manifest)} fixtures classically...\n")

for entry in manifest:
    filepath = os.path.join(out_dir, entry["file"])
    
    # 1. Load each QR with read_qr and decode the payload
    qr_string = read_qr_code(filepath)
    payload = decode_payload(qr_string)
    
    # 2. Recompute the expected tag and the secret against the REAL key
    expected_tag = compute_tag(key, payload["data"], payload["nonce"], n_bits)
    computed_secret = tags_to_secret(payload["tag"], expected_tag)
    
    # 3. Assert the computed secret matches the manifest's expected_secret
    assert computed_secret == entry["expected_secret"], (
        f"Manifest mismatch in {entry['file']}!\n"
        f"Expected: {entry['expected_secret']}\n"
        f"Computed: {computed_secret}"
    )
    
    # 4. Assert authentic fixtures give all zeros and tampered ones give non-zero
    if entry["expected_verdict"] == "authentic":
        assert computed_secret == "0" * n_bits, (
            f"False positive in {entry['file']}! Authentic QR yielded a non-zero secret."
        )
    else:
        assert "1" in computed_secret, (
            f"False negative collision in {entry['file']}! Tampered QR yielded an all-zero secret."
        )
        
    print(f"✅ {entry['file']:<25} | Verdict: {entry['expected_verdict']:<10} | Secret: {computed_secret}")

print("\n🎉 SUCCESS! All fixtures are internally consistent.")
print("The classical baseline is verified. The corpus is ready to test the quantum oracle!")

🔍 Sanity checking 5 fixtures classically...

✅ fixture_00_authentic.png  | Verdict: authentic  | Secret: 00000000
✅ fixture_01_data.png       | Verdict: tampered   | Secret: 00001010
✅ fixture_02_nonce.png      | Verdict: tampered   | Secret: 01010011
✅ fixture_03_tag.png        | Verdict: tampered   | Secret: 10000000
✅ fixture_04_forged.png     | Verdict: tampered   | Secret: 10111111

🎉 SUCCESS! All fixtures are internally consistent.
The classical baseline is verified. The corpus is ready to test the quantum oracle!


In [2]:
from quantum_qr.config import get_key
from quantum_qr.fixtures import build_fixture_set

# Define the exact same path
out_dir = "../data/fixtures"
key = get_key()

print("Generating tamper corpus... This might take a moment to compute.")

# Actually call the function to create the files!
manifest = build_fixture_set(key=key, out_dir=out_dir, n_bits=8)

print(f"✅ Successfully generated {len(manifest)} fixtures and saved manifest.json to {out_dir}!")

Generating tamper corpus... This might take a moment to compute.
✅ Successfully generated 5 fixtures and saved manifest.json to ../data/fixtures!
